In [1]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import data.cfr_data_19_23 as cfrd
import pandas as pd
import cfr.cfr_viz_helpers as vh
import plotly.express as px
import plotly.graph_objs as go
from scipy.stats import spearmanr, permutation_test

In [2]:
df = bd.load_meas_from_excel(
    # "infer_all_19_data_with_best_FEV1",
    "ppfev1st_ft_bFEV1_2016-19_IV_2019-21_assoc",
    study_folder="CFR",
    str_cols_to_arrays=[
        # "Airway resistance (%)",
        # "P(HFEV1|FEF2575, bFEV1, FEV1)",
        "P(HFEV1|bFEV1)",
        "P(HFEV1|FEV1)",
    ],
    use_csv=True,
    bypass_sanity_checks=True,
)

# Viz ranked associations with IV days

In [40]:
diff_col = "ppFEV1FT - ppFEV1ST"
prctile = 0  # keep rows where abs(diff_col) > this percentile threshold

df_ranked = df.copy()
# df_ranked = df_ranked[(df_ranked["ecFEF2575%ecFEV1"] < 70)].copy()
threshold = df_ranked[diff_col].abs().quantile(prctile / 100)
df_ranked = df_ranked[df_ranked[diff_col].abs() > threshold]

df_ranked.loc[df_ranked["FEV1%PredST"] >= 70, "severity"] = "mild"
df_ranked.loc[(df_ranked["FEV1%PredST"] >= 40) & (df_ranked["FEV1%PredST"] < 70), "severity"] = "moderate"
df_ranked.loc[df_ranked["FEV1%PredST"] < 40, "severity"] = "severe"

iv_max = df_ranked["IV days"].max() * 1.05

severities = [("mild", 1, 2), ("moderate", 3, 4), ("severe", 5, 6)]
fev_metrics = ["FEV1%PredST", "FEV1%PredFT"]
fev_colors = {"FEV1%PredST": "blue", "FEV1%PredFT": "red"}
iv_color = "rgba(64, 64, 64, 0.8)"

# Compute shared y range per severity across both fev_metrics
fev_range_by_severity = {}
for severity_label, _, _ in severities:
    df_sev = df_ranked[df_ranked["severity"] == severity_label]
    vals = pd.concat([df_sev[m] for m in fev_metrics]).dropna()
    pad = (vals.max() - vals.min()) * 0.05
    fev_range_by_severity[severity_label] = [vals.min() - pad, vals.max() + pad]


def add_ranked_fev_iv_panels(fig, df_severity, fev_metric, severity_label, row_fev, row_iv, col, overlay_metric=None):
    df_sorted = df_severity.sort_values(fev_metric, ascending=False).reset_index(drop=True)
    x_rank = list(range(len(df_sorted)))

    if overlay_metric is not None:
        fig.add_trace(
            go.Scatter(
                x=x_rank, y=df_sorted[overlay_metric],
                mode="markers",
                marker=dict(size=3, opacity=0.5, color=fev_colors[overlay_metric]),
                customdata=df_sorted["ID"],
                hovertemplate="ID: %{customdata}<br>" + overlay_metric + ": %{y:.1f}<extra></extra>",
                showlegend=False,
            ),
            row=row_fev, col=col,
        )

    fig.add_trace(
        go.Scatter(
            x=x_rank, y=df_sorted[fev_metric],
            mode="markers",
            marker=dict(size=3, opacity=1.0, color=fev_colors[fev_metric]),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>" + fev_metric + ": %{y:.1f}<extra></extra>",
            showlegend=False,
        ),
        row=row_fev, col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=x_rank, y=df_sorted["IV days"],
            mode="markers",
            marker=dict(size=3, color=iv_color),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>IV days: %{y:.0f}<extra></extra>",
            showlegend=False,
        ),
        row=row_iv, col=col,
    )
    fig.update_xaxes(showticklabels=False, linecolor="black", linewidth=1, showline=True, row=row_fev, col=col)
    fig.update_xaxes(showticklabels=False, linecolor="black", linewidth=1, showline=True, row=row_iv, col=col)
    fig.update_yaxes(title_text=f"{fev_metric}<br>{severity_label} CF", linecolor="black", linewidth=1, showline=True, range=fev_range_by_severity[severity_label], row=row_fev, col=col)
    fig.update_yaxes(title_text="IV days", range=[-5, iv_max], linecolor="black", linewidth=1, showline=True, row=row_iv, col=col)



fig = make_subplots(
    rows=6, cols=2,
    vertical_spacing=0.03,
    horizontal_spacing=0.12,
    column_titles=fev_metrics,
)

for col_idx, fev_metric in enumerate(fev_metrics, start=1):
    overlay = "FEV1%PredST" if col_idx == 2 else None
    for severity_label, row_fev, row_iv in severities:
        df_sev = df_ranked[df_ranked["severity"] == severity_label]
        add_ranked_fev_iv_panels(fig, df_sev, fev_metric, severity_label, row_fev, row_iv, col_idx)
        # add_ranked_fev_iv_panels(fig, df_sev, fev_metric, severity_label, row_fev, row_iv, col_idx, overlay_metric=overlay)

title = f"Ranked FEV1 and IV days by severity (|{diff_col}| > {prctile}th pctile)"
fig.update_layout(
    height=1100, width=900,
    title=title,
    template="simple_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.write_image(dh.get_path_to_main() + f"PlotsCFR/{title}.pdf")

# Correlations computation

In [42]:
df.columns

Index(['ID', 'Age', 'Height', 'FEV1', 'FEF2575', 'best FEV1', 'Sex',
       'Date Recorded', 'ecFEV1', 'ecFEF2575', 'ecFEF2575%ecFEV1',
       'Predicted FEV1', 'ecFEV1 % Predicted', 'FEV1 % Predicted',
       'best FEV1 old', 'idx FEV1', 'idx FEF2575%FEV1', 'idx best FEV1',
       'best FEV1 2016-19', 'best FEV1 year', 'bFEV1 diff', 'bFEV1 % diff',
       'idx best FEV1 2016-19', 'P(HFEV1|FEV1)', 'P(HFEV1|bFEV1)',
       'FEV1%PredST', 'FEV1%PredFT', 'ppFEV1FT - ppFEV1ST', 'IVs', 'IV days'],
      dtype='object')

In [ ]:
import numpy as np
from scipy import stats

# --- Configuration ---
baseline_col = "FEV1%PredST"   # baseline FEV metric; can also be "FEV1%Pred"
predicted_col = "FEV1%PredFT"  # always FEV1%PredFT
iv_col = "IV days"
diff_col = "ppFEV1FT - ppFEV1ST"
percentile_thresholds = [0, 25, 50, 75, 90]
N_BOOTSTRAP = 10_000

# Severity group definitions (based on baseline FEV1%PredST)
severity_groups = [
    ("Severe (<40)",        df["FEV1%PredST"] < 40),
    ("Moderate (40-69)", (df["FEV1%PredST"] >= 40) & (df["FEV1%PredST"] < 70)),
    # ("Moderate 2 (40-49)", (df["FEV1%PredST"] >= 40) & (df["FEV1%PredST"] < 50)),
    # ("Moderate 1 (50-69)", (df["FEV1%PredST"] >= 50) & (df["FEV1%PredST"] < 70)),
    # ("Mild 3 (70-79)",     (df["FEV1%PredST"] >= 70) & (df["FEV1%PredST"] < 80)),
    # ("Mild 2 (80-89)",     (df["FEV1%PredST"] >= 80) & (df["FEV1%PredST"] < 90)),
    # ("Mild 1 (>=90)",       df["FEV1%PredST"] >= 90),
    ("Mild 1 (>=90)",       df["FEV1%PredST"] >= 70),
]


def bootstrap_corr_diff(x, y, z, n_bootstrap=10_000, seed=42):
    """
    Bootstrap the difference between two one-sided Spearman correlations:
      diff = r(x, z) - r(y, z)

    x: baseline FEV1 values
    y: predicted FEV1 values (FEV1%PredFT)
    z: IV days

    If 0 is not in the returned CI, the difference is significant.
    """
    rng = np.random.default_rng(seed)
    x, y, z = np.array(x), np.array(y), np.array(z)
    n = len(x)

    res_x = stats.spearmanr(x, z, alternative="less")
    res_y = stats.spearmanr(y, z, alternative="less")
    obs_diff = res_x.statistic - res_y.statistic

    boot_diffs = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        boot_diffs[i] = (
            stats.spearmanr(x[idx], z[idx], alternative="less").statistic
            - stats.spearmanr(y[idx], z[idx], alternative="less").statistic
        )

    ci_lo_95 = np.percentile(boot_diffs, 2.5)
    ci_hi_95 = np.percentile(boot_diffs, 97.5)
    ci_lo_90 = np.percentile(boot_diffs, 5.0)
    ci_hi_90 = np.percentile(boot_diffs, 95.0)

    return {
        "r_baseline": res_x.statistic,
        "p_baseline": res_x.pvalue,
        "r_predicted": res_y.statistic,
        "p_predicted": res_y.pvalue,
        "diff": obs_diff,
        "ci_lo_95": ci_lo_95,
        "ci_hi_95": ci_hi_95,
        "significant_95": not (ci_lo_95 <= 0 <= ci_hi_95),
        "ci_lo_90": ci_lo_90,
        "ci_hi_90": ci_hi_90,
        "significant_90": not (ci_lo_90 <= 0 <= ci_hi_90),
        "n": n,
    }


def _analyze_subset(df_sub, label=""):
    n = len(df_sub)
    if n < 5:
        print(f"  [{label}] n={n}: skipping (too few samples)")
        return None
    res = bootstrap_corr_diff(
        df_sub[baseline_col], df_sub[predicted_col], df_sub[iv_col],
        n_bootstrap=N_BOOTSTRAP,
    )
    sig_95 = "SIG**" if res["significant_95"] else "ns"
    sig_90 = "SIG*"  if res["significant_90"] else "ns"
    print(
        f"  [{label}] n={res['n']:3d} | "
        f"r_base={res['r_baseline']:+.3f}(p={res['p_baseline']:.2e}), "
        f"r_pred={res['r_predicted']:+.3f}(p={res['p_predicted']:.2e}) | "
        f"diff={res['diff']:+.3f} | "
        f"95%CI=[{res['ci_lo_95']:+.3f},{res['ci_hi_95']:+.3f}] {sig_95} | "
        f"90%CI=[{res['ci_lo_90']:+.3f},{res['ci_hi_90']:+.3f}] {sig_90}"
    )
    return res


# --- Main Analysis ---
# all_results[prctile][label] = result dict
all_results = {}

for prctile in percentile_thresholds:
    print(f"\n{'='*100}")
    print(f"PERCENTILE THRESHOLD OF {diff_col}: {prctile}%")
    print(f"{'='*100}")
    all_results[prctile] = {}

    # Population level: threshold applied globally across all patients
    t_pop = df[diff_col].abs().quantile(prctile / 100)
    df_pop = df[df[diff_col].abs() >= t_pop].copy()
    print(f"\n  Population: abs({diff_col}) >= {t_pop:.2f}, n={len(df_pop)}")
    all_results[prctile]["Population"] = _analyze_subset(df_pop, label="Population")

    # By severity group: threshold applied within each group independently
    for group_name, group_mask in severity_groups:
        df_grp = df[group_mask].copy()
        if df_grp.empty:
            continue
        t_grp = df_grp[diff_col].abs().quantile(prctile / 100)
        df_sub = df_grp[df_grp[diff_col].abs() >= t_grp].copy()
        all_results[prctile][group_name] = _analyze_subset(df_sub, label=group_name)


# Viz baseline-prediction diff vs IV days

In [107]:
diff_col = "ppFEV1FT - ppFEV1ST"

dftmp = df.copy()

# Filter percentile of certain population
prctile = 90
for prctile in [50, 60, 70, 80, 90]:
    t = dftmp[diff_col].abs().quantile(prctile / 100)
    print(f"{prctile}th percentile of absolute difference: {t}")

    dftmp = dftmp[dftmp[diff_col].abs() > t]

    # Filter by severity level
    dftmp.loc[dftmp["FEV1%PredST"] >= 70, "severity"] = "mild"
    dftmp.loc[(dftmp["FEV1%PredST"] >= 40) & (dftmp["FEV1%PredST"] < 70), "severity"] = (
        "moderate"
    )
    dftmp.loc[dftmp["FEV1%PredST"] < 40, "severity"] = "severe"

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True)

    for i, severity in enumerate(["mild", "moderate", "severe"], start=1):
        dftmp_severity = dftmp[dftmp["severity"] == severity]
        fig.add_trace(
            go.Scatter(
                x=dftmp_severity[diff_col], y=dftmp_severity["IV days"], mode="markers", marker=dict(size=3, opacity=0.8)
            ),
            row=i,
            col=1,
        )
        fig.update_yaxes(title="IV days", range=[-5, dftmp["IV days"].max()*1.1], row=i, col=1)
    fig.update_xaxes(title=f"{diff_col}", row=3, col=1)
        

    title = f"bFEV1_2016_19_IVdays_2019-21_{prctile}th_prctile"
    fig.update_layout(
        height=600, width=800, title=title,
    )

    fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Viz diff vs IV days/{title}.pdf")

##########################################################################################
    def labels_from_bins(bins):
        labels = [f"< {bins[1]}"]
        for i in range(1, len(bins) - 2):
            labels.append(f"[{bins[i]}, {bins[i+1]})")
        labels.append(f">= {bins[-2]}")
        return labels


    fig = make_subplots(3, 1)

    row = 0
    for severity in ["mild", "moderate", "severe"]:
        row += 1
        dftmp2 = dftmp[dftmp["severity"] == severity].copy()
        bins = [-1000, -20, -11, -9, -7, -5, -3, -1, 0]
        labels = labels_from_bins(bins)
        dftmp2["diff_bin"] = pd.cut(dftmp2[diff_col], bins=bins, labels=labels)

        grouped = (
            dftmp2.groupby("diff_bin", observed=False)
            .agg(
                iv_mean=("IV days", "mean"),
                iv_std=("IV days", "std"),
                diff_mean=(diff_col, "mean"),
                count=(diff_col, "size"),
            )
            .reset_index()
        )

        fig.add_trace(
            go.Bar(
                x=grouped["diff_bin"].astype(str),
                y=grouped["iv_mean"],
                # mode="markers+text",
                error_y=dict(type="data", array=grouped["iv_std"].tolist(), visible=True),
                text=grouped["count"].astype(int),
                # textposition="top center",
                name=severity,
            ),
            row=row,
            col=1,
        )
        fig.update_yaxes(title="IV days (mean ± SD)", row=row, col=1)
    fig.update_xaxes(title=f"{diff_col} binned", row=1, col=1)


    title = f"bFEV1_2016_19_IVdays_2019-21_binned_{prctile}th_prctile"

    fig.update_layout(
        xaxis_title=diff_col,
        title=title,
        height=800,
    )
    fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Viz diff vs IV days/{title}.pdf")

50th percentile of absolute difference: 0.457945218826417
60th percentile of absolute difference: 2.781535403334735
70th percentile of absolute difference: 5.978852663315695
80th percentile of absolute difference: 12.69587666933772
90th percentile of absolute difference: 25.269813601145305
